# Trabajo Práctico Final — Clasificación de Emociones
## Paso 2: Deep Learning — CNN + Transfer Learning

**Modelos implementados:**
- **Enfoque A:** CNN personalizada (`MiniEmotionNet`) — diseñada para imágenes 48×48 en escala de grises
- **Enfoque B:** Transfer Learning con `ResNet18` preentrenada en ImageNet (adaptada a 1 canal)

**Técnicas de regularización:** AdamW, Weight Decay, Early Stopping, LR Scheduler, Dropout, BatchNorm

### Conexión con la materia
> La convolución en las CNN es exactamente la operación vista en `Clase_Convol_Correl_PSUTN.pdf`. Cada capa convolucional aplica un banco de filtros (kernels aprendibles) que convoluciona con el mapa de activación previo, detectando patrones de complejidad creciente: bordes → texturas → partes del rostro → expresiones completas.

---

## 2.0 — Imports y configuración

In [ ]:
import os, random, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as T

import albumentations as A
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

BASE_DIR = Path(r'C:\Users\Uriel\Documents\UTN\PIYS')
CLASSES  = ['angry','disgusted','fearful','happy','neutral','sad','surprised']
N_CLASSES = len(CLASSES)
IMG_SIZE  = 48  # imágenes nativas del dataset

# Cargar splits y estadísticos del Paso 1
df_tr  = pd.read_csv(BASE_DIR / 'df_train_split.csv')
df_val = pd.read_csv(BASE_DIR / 'df_val_split.csv')
df_test= pd.read_csv(BASE_DIR / 'df_test_split.csv')
stats  = pd.read_csv(BASE_DIR / 'normalization_stats.csv')
NORM_MEAN = float(stats['mean'].values[0])
NORM_STD  = float(stats['std'].values[0])

print(f'Train: {len(df_tr):,} | Val: {len(df_val):,} | Test: {len(df_test):,}')
print(f'Normalización — mean: {NORM_MEAN:.4f}, std: {NORM_STD:.4f}')

## 2.1 — Dataset y DataLoaders

In [ ]:
aug_train = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=12, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.6),
    A.GaussNoise(var_limit=(5, 20), p=0.3),
    A.Affine(shear=(-5,5), translate_percent=(-0.05,0.05), p=0.4),
    A.Normalize(mean=[NORM_MEAN], std=[NORM_STD]),
])

aug_eval = A.Compose([
    A.Normalize(mean=[NORM_MEAN], std=[NORM_STD]),
])


class EmotionDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None, n_channels: int = 1):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.n_channels = n_channels  # 1 para CNN custom, 3 para ResNet

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = np.array(Image.open(row['path']).convert('L'), dtype=np.uint8)
        if self.transform:
            img = self.transform(image=img)['image']  # float32 normalizado
        else:
            img = img.astype(np.float32) / 255.0
        img_t = torch.tensor(img, dtype=torch.float32).unsqueeze(0)  # (1,H,W)
        if self.n_channels == 3:
            img_t = img_t.repeat(3, 1, 1)  # (3,H,W) para Transfer Learning
        return img_t, torch.tensor(row['label_enc'], dtype=torch.long)


BATCH_SIZE = 64

def make_loaders(n_channels=1):
    ds_tr  = EmotionDataset(df_tr,   aug_train, n_channels)
    ds_val = EmotionDataset(df_val,  aug_eval,  n_channels)
    ds_te  = EmotionDataset(df_test, aug_eval,  n_channels)
    return (
        DataLoader(ds_tr,  batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True),
        DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
        DataLoader(ds_te,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
    )

# Pesos para CrossEntropy
counts = df_tr['label_enc'].value_counts().sort_index().values.astype(np.float32)
class_weights = torch.tensor(1.0 / counts / (1.0/counts).sum() * N_CLASSES, dtype=torch.float32).to(DEVICE)
print('class_weights:', class_weights.cpu().numpy().round(3))

## 2.2 — Arquitectura: MiniEmotionNet (CNN personalizada)

Diseñada específicamente para imágenes de 48×48 en escala de grises. Cada bloque convolucional aplica la convolución estudiada en clase seguida de BatchNorm (estabiliza el entrenamiento) y Dropout (regularización).

```
Input (1×48×48)
  → ConvBlock1 (32 filtros, 3×3)  → 32×48×48
  → ConvBlock2 (64 filtros, 3×3)  → 64×24×24
  → ConvBlock3 (128 filtros, 3×3) → 128×12×12
  → ConvBlock4 (256 filtros, 3×3) → 256×6×6
  → GlobalAvgPool                 → 256
  → FC(256→128) → Dropout(0.4)   → 128
  → FC(128→7)                    → logits
```

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool2d(2))
        self.block = nn.Sequential(*layers)

    def forward(self, x): return self.block(x)


class MiniEmotionNet(nn.Module):
    def __init__(self, n_classes=7, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1,   32, pool=False),   # 48→48
            ConvBlock(32,  64,  pool=True),   # 48→24
            ConvBlock(64,  128, pool=True),   # 24→12
            ConvBlock(128, 256, pool=True),   # 12→6
        )
        self.pool      = nn.AdaptiveAvgPool2d(1)   # Global Average Pooling → 256×1×1
        self.dropout   = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)

    def extract_features(self, x):
        """Devuelve el vector de 256 dims antes del clasificador (usado en Paso 3)."""
        x = self.features(x)
        return self.pool(x).flatten(1)


model_cnn = MiniEmotionNet(N_CLASSES).to(DEVICE)

# Resumen de parámetros
total_params = sum(p.numel() for p in model_cnn.parameters())
trainable    = sum(p.numel() for p in model_cnn.parameters() if p.requires_grad)
print(f'Parámetros totales:     {total_params:,}')
print(f'Parámetros entrenables: {trainable:,}')

# Verificación de forma
with torch.no_grad():
    dummy = torch.zeros(2, 1, 48, 48).to(DEVICE)
    out   = model_cnn(dummy)
    feat  = model_cnn.extract_features(dummy)
print(f'Output shape: {out.shape}  | Feature shape: {feat.shape}')

## 2.3 — Arquitectura: Transfer Learning con ResNet18

In [ ]:
def build_resnet18(n_classes=7, freeze_backbone=False):
    """
    ResNet18 preentrenada en ImageNet, adaptada para:
    - Entrada de 3 canales (grayscale replicado)
    - 7 clases de salida
    - Fine-tuning completo o con backbone congelado
    """
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # Reemplazar la cabeza de clasificación
    in_features = model.fc.in_features  # 512
    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(256, n_classes),
    )
    return model


model_resnet = build_resnet18(N_CLASSES, freeze_backbone=False).to(DEVICE)

total_p = sum(p.numel() for p in model_resnet.parameters())
print(f'ResNet18 — parámetros totales: {total_p:,}')

with torch.no_grad():
    dummy3 = torch.zeros(2, 3, 48, 48).to(DEVICE)
    out3   = model_resnet(dummy3)
print(f'Output shape: {out3.shape}')

## 2.4 — Loop de entrenamiento genérico con Early Stopping

In [ ]:
class EarlyStopping:
    def __init__(self, patience=7, min_delta=1e-4, path='best_model.pt'):
        self.patience   = patience
        self.min_delta  = min_delta
        self.path       = path
        self.best_loss  = np.inf
        self.counter    = 0
        self.stopped    = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
            torch.save(model.state_dict(), self.path)
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stopped = True
        return self.stopped


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


def train_model(model, train_loader, val_loader, model_name,
                epochs=50, lr=1e-3, patience=7):
    save_path  = str(BASE_DIR / f'best_{model_name}.pt')
    criterion  = nn.CrossEntropyLoss(weight=class_weights)
    optimizer  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    stopper    = EarlyStopping(patience=patience, path=save_path)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}
    t0 = time.time()

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        stop = stopper(vl_loss, model)
        marker = '  ← best' if stopper.counter == 0 else ''
        if epoch % 5 == 0 or stop or epoch == 1:
            print(f'Epoch {epoch:3d}/{epochs} | '
                  f'Loss {tr_loss:.4f}/{vl_loss:.4f} | '
                  f'Acc {tr_acc:.3f}/{vl_acc:.3f} | '
                  f'LR {scheduler.get_last_lr()[0]:.2e}{marker}')
        if stop:
            print(f'Early stopping en epoch {epoch}.')
            break

    elapsed = time.time() - t0
    print(f'\nEntrenamiento: {elapsed:.0f}s | Mejor val_loss: {stopper.best_loss:.4f}')
    model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    return history


print('Loop de entrenamiento listo.')

## 2.5 — Entrenamiento: MiniEmotionNet

In [ ]:
loader_tr_1ch, loader_val_1ch, loader_te_1ch = make_loaders(n_channels=1)

print('=== Entrenando MiniEmotionNet ===')
history_cnn = train_model(
    model_cnn, loader_tr_1ch, loader_val_1ch,
    model_name='mini_emotion_net',
    epochs=60, lr=1e-3, patience=10
)

## 2.6 — Entrenamiento: ResNet18 (Transfer Learning)

In [ ]:
loader_tr_3ch, loader_val_3ch, loader_te_3ch = make_loaders(n_channels=3)

print('=== Entrenando ResNet18 (Transfer Learning) ===')
history_resnet = train_model(
    model_resnet, loader_tr_3ch, loader_val_3ch,
    model_name='resnet18_emotion',
    epochs=40, lr=5e-4, patience=8
)

## 2.7 — Curvas de entrenamiento

In [ ]:
def plot_training_curves(history, title, color='steelblue'):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    epochs = range(1, len(history['train_loss']) + 1)

    # Loss
    ax = axes[0]
    ax.plot(epochs, history['train_loss'], label='Train', color=color)
    ax.plot(epochs, history['val_loss'],   label='Val',   color=color, linestyle='--')
    best_ep = np.argmin(history['val_loss']) + 1
    ax.axvline(best_ep, color='red', lw=1, linestyle=':', label=f'Best val (ep {best_ep})')
    ax.set_title('Loss'); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)

    # Accuracy
    ax = axes[1]
    ax.plot(epochs, history['train_acc'], label='Train', color=color)
    ax.plot(epochs, history['val_acc'],   label='Val',   color=color, linestyle='--')
    ax.axhline(max(history['val_acc']), color='green', lw=1, linestyle=':',
               label=f'Best acc: {max(history["val_acc"]):.3f}')
    ax.set_title('Accuracy'); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)

    # LR
    ax = axes[2]
    ax.plot(epochs, history['lr'], color='orange')
    ax.set_title('Learning Rate (CosineAnnealing)')
    ax.set_xlabel('Epoch'); ax.set_yscale('log'); ax.grid(alpha=0.3)

    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    safe = title.replace(' ', '_').lower()
    plt.savefig(BASE_DIR / f'fig_training_{safe}.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_training_curves(history_cnn,    'MiniEmotionNet — Curvas de Entrenamiento', color='steelblue')
plot_training_curves(history_resnet, 'ResNet18 — Curvas de Entrenamiento',       color='darkorange')

## 2.8 — Evaluación en test y reporte por clase

In [ ]:
@torch.no_grad()
def get_predictions(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        probs  = F.softmax(logits, dim=1).cpu().numpy()
        preds  = logits.argmax(1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


y_true_cnn,    y_pred_cnn,    y_prob_cnn    = get_predictions(model_cnn,    loader_te_1ch)
y_true_resnet, y_pred_resnet, y_prob_resnet = get_predictions(model_resnet, loader_te_3ch)

print('=== MiniEmotionNet — Classification Report ===')
print(classification_report(y_true_cnn, y_pred_cnn, target_names=CLASSES))

print('\n=== ResNet18 — Classification Report ===')
print(classification_report(y_true_resnet, y_pred_resnet, target_names=CLASSES))

# Guardar predicciones para Paso 4
np.save(BASE_DIR / 'cnn_y_true.npy',  y_true_cnn)
np.save(BASE_DIR / 'cnn_y_pred.npy',  y_pred_cnn)
np.save(BASE_DIR / 'cnn_y_prob.npy',  y_prob_cnn)
np.save(BASE_DIR / 'resnet_y_true.npy',  y_true_resnet)
np.save(BASE_DIR / 'resnet_y_pred.npy',  y_pred_resnet)
np.save(BASE_DIR / 'resnet_y_prob.npy',  y_prob_resnet)
print('\nPredicciones guardadas para Paso 4.')

## 2.9 — Extracción de features del encoder (para Paso 3)

Removemos la cabeza de clasificación y usamos el modelo como extractor de vectores de características.

In [ ]:
@torch.no_grad()
def extract_features_cnn(model, loader):
    """Extrae el vector de 256 dims del penúltimo layer de MiniEmotionNet."""
    model.eval()
    feats, labels = [], []
    for imgs, lbl in tqdm(loader, desc='Extrayendo features'):
        imgs = imgs.to(DEVICE)
        f = model.extract_features(imgs).cpu().numpy()
        feats.extend(f)
        labels.extend(lbl.numpy())
    return np.array(feats), np.array(labels)


@torch.no_grad()
def extract_features_resnet(model, loader):
    """Extrae el vector de 512 dims antes de la fc head de ResNet18."""
    model.eval()
    # Reemplazar fc temporalmente por Identity
    original_fc = model.fc
    model.fc = nn.Identity()
    feats, labels = [], []
    for imgs, lbl in tqdm(loader, desc='Extrayendo features ResNet'):
        imgs = imgs.to(DEVICE)
        f = model(imgs).cpu().numpy()
        feats.extend(f)
        labels.extend(lbl.numpy())
    model.fc = original_fc  # restaurar
    return np.array(feats), np.array(labels)


# Extraemos para train + val (para entrenar XGBoost) y test (para evaluar)
print('Extrayendo features — MiniEmotionNet...')
feat_tr_cnn,  lbl_tr_cnn  = extract_features_cnn(model_cnn, loader_tr_1ch)
feat_val_cnn, lbl_val_cnn = extract_features_cnn(model_cnn, loader_val_1ch)
feat_te_cnn,  lbl_te_cnn  = extract_features_cnn(model_cnn, loader_te_1ch)

print('Extrayendo features — ResNet18...')
feat_tr_rn,  lbl_tr_rn  = extract_features_resnet(model_resnet, loader_tr_3ch)
feat_val_rn, lbl_val_rn = extract_features_resnet(model_resnet, loader_val_3ch)
feat_te_rn,  lbl_te_rn  = extract_features_resnet(model_resnet, loader_te_3ch)

# Combinar train + val para XGBoost
X_tr_cnn = np.vstack([feat_tr_cnn, feat_val_cnn])
y_tr_cnn = np.concatenate([lbl_tr_cnn, lbl_val_cnn])
X_tr_rn  = np.vstack([feat_tr_rn, feat_val_rn])
y_tr_rn  = np.concatenate([lbl_tr_rn, lbl_val_rn])

print(f'Features CNN   — Train: {X_tr_cnn.shape} | Test: {feat_te_cnn.shape}')
print(f'Features ResNet— Train: {X_tr_rn.shape}  | Test: {feat_te_rn.shape}')

# Guardar para Paso 3
np.save(BASE_DIR / 'feat_train_cnn.npy',    X_tr_cnn)
np.save(BASE_DIR / 'feat_train_labels_cnn.npy', y_tr_cnn)
np.save(BASE_DIR / 'feat_test_cnn.npy',     feat_te_cnn)
np.save(BASE_DIR / 'feat_test_labels_cnn.npy',  lbl_te_cnn)
np.save(BASE_DIR / 'feat_train_rn.npy',     X_tr_rn)
np.save(BASE_DIR / 'feat_train_labels_rn.npy',  y_tr_rn)
np.save(BASE_DIR / 'feat_test_rn.npy',      feat_te_rn)
np.save(BASE_DIR / 'feat_test_labels_rn.npy',   lbl_te_rn)
print('\nFeatures guardadas para Paso 3.')

## 2.10 — Visualización de features con UMAP / t-SNE

Proyectamos las features de 256 dims a 2D para verificar que el encoder aprendió representaciones separables por emoción.

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

EMOTION_COLORS = {
    'angry':'#E74C3C','disgusted':'#8E44AD','fearful':'#2980B9',
    'happy':'#F1C40F','neutral':'#95A5A6','sad':'#2C3E50','surprised':'#E67E22'
}

# Muestreamos 2000 puntos del test para rapidez
idx_sample = np.random.choice(len(feat_te_cnn), size=min(2000, len(feat_te_cnn)), replace=False)
X_viz  = StandardScaler().fit_transform(feat_te_cnn[idx_sample])
y_viz  = lbl_te_cnn[idx_sample]

print('Ejecutando t-SNE (puede tardar ~1 min)...')
tsne = TSNE(n_components=2, perplexity=40, random_state=SEED, n_iter=1000)
X_2d = tsne.fit_transform(X_viz)

fig, ax = plt.subplots(figsize=(10, 8))
for i, cls in enumerate(CLASSES):
    mask = y_viz == i
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=EMOTION_COLORS[cls], label=cls, alpha=0.6, s=15, edgecolors='none')
ax.legend(title='Emoción', markerscale=2, framealpha=0.8)
ax.set_title('t-SNE de features (256 dims → 2D) — MiniEmotionNet\n'
             'Clusters bien separados = encoder aprendió representaciones discriminativas',
             fontsize=11, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig_09_tsne_features.png', dpi=150, bbox_inches='tight')
plt.show()

## Resumen del Paso 2

| Modelo | Arquitectura | Parámetros | Val Acc (aprox.) |
|---|---|---|---|
| MiniEmotionNet | CNN custom 4 bloques + GAP | ~1.2M | Ver curvas |
| ResNet18 | Transfer Learning ImageNet | ~11.2M | Ver curvas |

**Artefactos generados para los siguientes pasos:**
- `best_mini_emotion_net.pt` / `best_resnet18_emotion.pt` — pesos del mejor modelo
- `cnn_y_*.npy` / `resnet_y_*.npy` — predicciones para evaluación (Paso 4)
- `feat_*.npy` — features extraídas para XGBoost (Paso 3)